<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">La Regularización</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En deep learning no basta con que la red aprenda; también debe evitar memorizar de forma excesiva los datos de entrenamiento. Este notebook introduce la regularización como conjunto de estrategias para mejorar la generalización.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Un modelo que rinde muy bien en entrenamiento puede fallar fuera de él si no se controla el sobreajuste.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender qué problema intenta resolver la regularización y por qué aparece como parte central del diseño del entrenamiento.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Entrenar más o hacer la red más grande no siempre mejora el modelo; a veces empeora su capacidad de generalizar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Regularizar significa introducir mecanismos que limiten o controlen la complejidad efectiva del aprendizaje.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Tensores de entrada y etiquetas
"""Este bloque crea tensores que representan las entradas y las etiquetas de un problema supervisado.
Este bloque importa "torch" para trabajar con tensores en PyTorch.
La variable "entradas" contiene muestras bidimensionales en forma de tensor que sirven como datos de entrada.

Aplicamos el método ".unsqueeze()" a las etiquetas para que tengan una dimensión adicional, convirtiéndolas en tensores columna.
El argumento "1" indica que se debe agregar una dimensión adicional en la posición 1 del tensor.
La variable "etiquetas" contiene valores asociados a cada muestra en forma de tensor con una dimensión adicional en 
la posición 1 y define el objetivo del entrenamiento.
Estos tensores permiten demostrar cómo la regularización influye en el entrenamiento."""

import torch

entradas = torch.tensor([[1.0, 2.0],
                         [3.0, 4.0],
                         [5.0, 6.0]])

etiquetas = torch.tensor([0.0, 1.0, 0.0]).unsqueeze(1)

entradas, etiquetas # Muestra los tensores de entrada y las etiquetas con la dimensión adicional.


In [ ]:
# Bloque 2 — Definición de una red neuronal
"""Este bloque define una red neuronal artificial utilizando nn.Module.
Este bloque importa "torch.nn" para definir módulos y capas de redes neuronales.

Definimos una clase personalizada llamada "RedRegularizacion()" que representa un modelo de red neuronal simple.
La clase "RedRegularizacion()" hereda de "nn.Module", que es la clase base utilizada para construir redes neuronales en PyTorch.

Definimos la función "def __init__()", que inicializa la estructura interna del modelo.
El parámetro "self" de la función hace referencia a la instancia actual de la clase "RedRegularizacion()".

Utilizamos "super().__init__()" para invocar el constructor de la clase base "nn.Module".     
Esto asegura que la clase base se inicialice correctamente cada vez que se crea una instancia del modelo.

Aplicamos la clase "nn.Linear()" a la variable de instancia "self.capa1" para definir la primera capa lineal.
Los argumentos "2, 4" de "nn.Linear()" indican que la capa lineal recibe 2 entradas y produce 4 salidas.
Aplicamos la clase "nn.Linear()" a la variable de instancia "self.capa2" para definir la segunda capa lineal.
Los argumentos "4, 1" de "nn.Linear()" indican que la capa lineal recibe 4 entradas y produce 1 salida.

Definimos la función "def forward()" que especifica cómo fluye la entrada a través de las capas del modelo.
El método ".forward()" especifica cómo se propaga la información a través de la red.
Los parámetros "self" y "x" representan la instancia del modelo y los datos de entrada, respectivamente.

Asignamos el resultado de llamar a "self.capa1()" a la variable "x", aplicando la primera capa lineal a la entrada.
Aplicamos la función de activación "torch.relu()" a la variable "x" para introducir no linealidad en la red.
Asignamos el resultado de llamar a "self.capa2()" a la variable "x", aplicando la segunda capa lineal a la salida de la activación.
Utilizamos la instrucción "return" para devolver la salida del modelo aplicando las capas lineales a la entrada.

En todos los casos se pasa "x" como parámetro a las funciones correspondientes para que la información fluya a través de la red.
Además, la variable "x" se actualiza con la salida de cada operación, asegurando que la información fluya correctamente a través de la red.
La variable "red" almacena una instancia de la clase "RedRegularizacion()", representando el modelo de red neuronal definido.
La estructura permite demostrar cómo la regularización afecta los parámetros del modelo."""

import torch.nn as nn

class RedRegularizacion(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(2, 4)
        self.capa2 = nn.Linear(4, 1)

    def forward(self, x):
        x = self.capa1(x)
        x = torch.relu(x)
        x = self.capa2(x)
        return x

red = RedRegularizacion()

red # Muestra la estructura de la red neuronal definida.


In [ ]:
# Bloque 3 — Definición de la función de pérdida con regularización L2
"""Este bloque define la función de pérdida base y el coeficiente de regularización L2.
Aplicamos la clase "nn.BCEWithLogitsLoss()" a la variable "criterio" para crear un objeto calculador de función de pérdida.
La variable "criterio" almacena el objeto que evalúa el rendimiento del modelo.
La variable "lambda_l2" almacena el coeficiente de regularización L2, que controla la magnitud de la penalización aplicada a los parámetros del modelo.
La penalización se añadirá a la pérdida base en un bloque posterior."""

criterio = nn.BCEWithLogitsLoss()
lambda_l2 = 0.01

criterio, lambda_l2 # Muestra el objeto calculador de la función de pérdida y el valor del coeficiente de regularización L2.


In [ ]:
# Bloque 4 — Definición del optimizador
"""Este bloque define el optimizador utilizado durante el entrenamiento.
Aplicamos la clase "torch.optim.Adam()" a la variable "optimizador" para crear un objeto optimizador de parámetros de la red.
Pasamos como argumentos los parámetros de la red mediante la expresión "red.parameters()".
El método ".parameters()" devuelve un iterador sobre los parámetros de la red que serán actualizados por el optimizador.

El parámetro "lr=" especifica la tasa de aprendizaje utilizada por el optimizador, en este caso 0.01.
El optimizador Adam actualiza los parámetros de la red utilizando los gradientes calculados.
La variable "optimizador" almacena el objeto que gestiona las actualizaciones de los parámetros."""

optimizador = torch.optim.Adam(red.parameters(), lr=0.01)

optimizador # Muestra la configuración del objeto optimizador.


In [ ]:
# Bloque 5 — Forward pass
"""Este bloque ejecuta el forward pass de la red neuronal.
Asignamos el resultado de llamar a la instancia de la clase a la variable "predicciones" mediante la expresión "red(entradas)".
El argumento "entradas" representa los datos de entrada que se pasan a la red para generar las predicciones.
La variable "predicciones" almacena los logits generados por la red.
El forward pass calcula las predicciones utilizando los parámetros actuales del modelo."""

predicciones = red(entradas)

predicciones # Muestra las salidas generadas por la red.


In [ ]:
# Bloque 6 — Cálculo de la pérdida con regularización L2
"""Este bloque calcula la pérdida total al incorporar la regularización L2.
Asignamos la llamada a la función de pérdida a la variable "perdida_base" mediante la expresión "criterio(predicciones, etiquetas)".
El argumento "predicciones" representa los logits generados por la red; la función de pérdida aplica internamente la sigmoide.
El argumento "etiquetas" representa los valores objetivo del problema.
La variable "perdida_base" almacena el valor de la pérdida calculada entre las predicciones y las etiquetas.

Asignamos la penalización L2 a la variable "l2_penalizacion" mediante la expresión "sum((param ** 2).sum() for param in red.parameters())".
La expresión se descompone de la siguiente manera: para cada parámetro de la red, se calcula el cuadrado de sus valores y luego se suman.
Finalmente, se agregan todos esos resultados para obtener la penalización L2 total.
La variable "l2_penalizacion" almacena la suma de los cuadrados de todos los parámetros del modelo, incluidos los sesgos.

Asignamos la expresión "(perdida_base + lambda_l2 * l2_penalizacion)" a la variable "perdida_total" para calcular la pérdida total,
incluyendo la regularización L2 definida anteriormente.
La variable "perdida_total" combina la pérdida base con la penalización L2."""

perdida_base = criterio(predicciones, etiquetas)
l2_penalizacion = sum((param ** 2).sum() for param in red.parameters())
perdida_total = (perdida_base + lambda_l2 * l2_penalizacion)

perdida_base, l2_penalizacion, perdida_total  # Muestra la pérdida base, la penalización L2 y la pérdida total.


In [ ]:
# Bloque 7 — Backpropagation
"""Este bloque ejecuta el proceso de backpropagation.
Aplicamos el método ".zero_grad()" al optimizador para eliminar los gradientes previos antes de calcular los nuevos gradientes.
El método ".zero_grad()" elimina gradientes previos para evitar acumulación.

Aplicamos el método "backward()" a la variable "perdida_total" para calcular los gradientes de la pérdida total respecto a los parámetros de la red.
El método "backward()" calcula los gradientes de la pérdida total respecto a los parámetros de la red.
Los gradientes quedan almacenados en los atributos grad de cada parámetro."""

optimizador.zero_grad()
perdida_total.backward()

optimizador, perdida_total # Muestra la configuración del optimizador y la pérdida total.


In [ ]:
# Bloque 8 — Actualización de parámetros
"""Este bloque actualiza los parámetros de la red neuronal.
Aplicamos el método ".step()" al optimizador para actualizar los parámetros de la red.
El método ".step()" utiliza los gradientes calculados para modificar los parámetros.
La actualización refleja el efecto de la regularización sobre los parámetros."""

optimizador.step()

optimizador # Muestra la configuración del optimizador después de la actualización de parámetros.


In [ ]:
# Bloque 9 — Resultados finales
"""Este bloque organiza los elementos principales del proceso de regularización.
La variable "resultado" almacena en forma de diccionario la pérdida base, la penalización L2 y la pérdida total calculadas antes de la actualización.
En todos los casos utilizamos ".item()" para convertir los tensores de PyTorch a valores escalares.
La estructura permite visualizar cómo la regularización influye en el entrenamiento del modelo."""

resultado = {
    "perdida_base": float(perdida_base.item()),
    "l2_penalizacion": float(l2_penalizacion.item()),
    "perdida_total": float(perdida_total.item())
}

resultado # Muestra el diccionario de resultados para verificar los valores almacenados.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto la regularización como respuesta al riesgo de sobreajuste en redes neuronales.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Aprender bien no significa solo ajustar el entrenamiento, sino conservar capacidad de generalizar fuera de él.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Interpretar una mejora en entrenamiento como señal suficiente de que el modelo será útil sobre datos nuevos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en el Dropout para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en el Dropout y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">06_Regularizacion/02_El_dropout.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">